In [ ]:
%load_ext autoreload
%autoreload 2

from spatial_tcr.utils import switch_cwd_to_root

switch_cwd_to_root()

import spatialtools as st

figure_dir = "figures/revision/rebuttal"
st.utils.setup_plotting(figure_dir)

import os

import nichepca as npc
import numpy as np
import pandas as pd
import scanpy as sc
from spatialtools.nhood_analysis import (
    compute_nhood_composition,
)
from spatialtools.permutation import permute_positions
from tqdm.auto import tqdm

from spatial_tcr.clonal_expansion import (
    filter_clonal_clusters_by_cell_type,
    identify_clonal_clusters,
    merge_clonal_clusters,
)
from spatial_tcr.tcr import get_tcr_genes

In [10]:
def compare_nhood_composition(
    adata,
    ct_key="cell_type_l2",
    cc_key="avbv_cluster_filtered",
):
    adata = adata.copy()
    adata.obs["is_avbv_cluster"] = adata.obs[cc_key].notna().astype(str)
    adata.obs.loc[adata.obs[cc_key].isna(), "is_avbv_cluster"] = None

    # prepare other T cell column that excludes all T cells in clusters
    adata.obs["ct_outside_cluster"] = adata.obs[ct_key].copy()
    adata.obs.loc[adata.obs[cc_key].notna(), "ct_outside_cluster"] = None

    npc.gc.construct_multi_sample_graph(
        adata,
        sample_key="sample",
        obsm_key="spatial",
        radius=25,
        remove_self_loops=True,
        verbose=False,
    )

    ad_nhood_tclonal = compute_nhood_composition(
        adata, obs_key="is_avbv_cluster", comp_key=ct_key
    )
    ad_nhood_not_tclonal = compute_nhood_composition(
        adata, obs_key="ct_outside_cluster", comp_key=ct_key
    )
    print("nhoods with tcells not in cluster (unfiltered)", ad_nhood_not_tclonal.shape)
    tcell_keys = [
        "CD4+",
        # "TFH",
        "Tregs",
        "MAIT",
        "CD8+",
        "NKT-like",
    ]
    ad_nhood_tcells_not_clonal = ad_nhood_not_tclonal[
        ad_nhood_not_tclonal.obs["ct_outside_cluster"].isin(tcell_keys)
    ].copy()

    print("nhoods with tcells in cluster", ad_nhood_tclonal.shape)
    print("nhoods with tcells not in cluster", ad_nhood_tcells_not_clonal.shape)

    # merge the expression
    var_names = ad_nhood_tclonal.var_names
    ad_merged = sc.AnnData(
        X=np.concatenate(
            [
                ad_nhood_tclonal.X,
                ad_nhood_tcells_not_clonal[:, var_names].X,
            ],
            axis=0,
        ),
        var=ad_nhood_tclonal.var,
        obs=pd.concat([ad_nhood_tclonal.obs, ad_nhood_tcells_not_clonal.obs], axis=0),
    )
    ad_merged.obs["group"] = "cluster\nnhood."
    ad_merged.obs.loc[ad_nhood_tcells_not_clonal.obs_names, "group"] = (
        "non-cluster\nnhood."
    )
    # display(ad_merged.obs["group"].value_counts(dropna=False))

    ct_comp_merged = ad_merged.to_df()
    ct_comp_merged = pd.concat([ct_comp_merged, ad_merged.obs[["group"]]], axis=1)

    # display(ct_comp_merged)

    group_key = "group"

    ct_comp_merged_normalized = ct_comp_merged.copy()
    ct_cols = [c for c in ct_comp_merged_normalized.columns if c != group_key]
    ct_comp_merged_normalized[ct_cols] = ct_comp_merged_normalized[ct_cols].div(
        ct_comp_merged_normalized[ct_cols].sum(axis=1), axis=0
    )

    # display(ct_comp_merged_normalized)

    mean_comp = ct_comp_merged_normalized.groupby(group_key).mean()
    # display(mean_comp)
    ratios = mean_comp.iloc[0] / mean_comp.iloc[1]
    ratios = ratios.sort_values(ascending=True)
    mean_comp = mean_comp[ratios.index]
    return mean_comp

In [3]:
data_dir = "data/xenium/processed"
path = f"{data_dir}/08.1-kidney_tcr_clonal_clusters.h5ad"
adata = sc.read_h5ad(path)
adata

AnnData object with n_obs × n_vars = 510139 × 431
    obs: 'sample', 'x_centroid', 'y_centroid', 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'unassigned_codeword_counts', 'deprecated_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'slide', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'log1p_total_counts', 'pct_counts_in_top_10_genes', 'pct_counts_in_top_20_genes', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_150_genes', 'n_counts', 'condition', 'cc', 'cell_type_no_tcr', 'cell_type_no_tcr_prob', 'tcell_subtype', 'cell_type_l1', 'cell_type_l2', 'is_ATL', 'is_B', 'is_CNT', 'is_DCT', 'is_DTL', 'is_EC', 'is_FIB', 'is_IC', 'is_MAST', 'is_MC', 'is_MDC', 'is_Mac', 'is_N', 'is_NEU', 'is_PC', 'is_PEC', 'is_PL', 'is_POD', 'is_PT', 'is_PapE', 'is_T', 'is_TAL', 'is_VSM/P', 'is_cDC', 'is_cycMNP', 'is_glom. EC', 'is_pDC', 'is_unknown', 'leiden', 'glom_annot', 'in_glom', 'tcell_density_group', 'tcell_density', 'tcell_infiltrate', 'cell_type_l1.1', 'av_

In [4]:
mean_comp = compare_nhood_composition(
    adata, ct_key="cell_type_l2", cc_key="avbv_cluster_filtered"
)
mean_comp.reset_index(inplace=True)
mean_comp["iter"] = 0
mean_comp

  0%|          | 0/13 [00:00<?, ?it/s]

,group,PapE,NEU,POD,PEC,MC,PC,DCT,pDC,glom. EC,...,gdT,unknown,MDC,MAIT,B,CD4+,CD8+,cycMNP,Tregs,iter
0,cluster\nnhood.,0.000000,0.000000,0.000682,0.000908,0.000289,0.002715,0.002452,0.000084,0.001737,...,0.003135,0.016791,0.131044,0.009515,0.075384,0.112351,0.075758,0.006774,0.018978,0
1,non-cluster\nnhood.,0.000084,0.000083,0.005599,0.005140,0.001345,0.010037,0.008292,0.000198,0.003555,...,0.002467,0.011756,0.085431,0.006093,0.047042,0.066610,0.042710,0.003452,0.008160,0


In [23]:
mean_comp

,group,PapE,NEU,POD,PEC,MC,PC,DCT,pDC,glom. EC,...,gdT,unknown,MDC,MAIT,B,CD4+,CD8+,cycMNP,Tregs,iter
0,cluster\nnhood.,0.000000,0.000000,0.000682,0.000908,0.000289,0.002715,0.002452,0.000084,0.001737,...,0.003135,0.016791,0.131044,0.009515,0.075384,0.112351,0.075758,0.006774,0.018978,0
1,non-cluster\nnhood.,0.000084,0.000083,0.005599,0.005140,0.001345,0.010037,0.008292,0.000198,0.003555,...,0.002467,0.011756,0.085431,0.006093,0.047042,0.066610,0.042710,0.003452,0.008160,0


In [6]:
av_genes, bv_genes = get_tcr_genes(adata)[0:2]

Found 35 TRAV genes, 31 TRBV genes, 3 TRDV genes, 14 TRGV genes


In [11]:
seed = 42
rng = np.random.RandomState(seed)
n_perms = 2
layer = "avbv_clean"
ct_key = "cell_type_l1.1"
tcell_keys = [
    "CD4+",
    "MAIT",
    "CD8+",
    "NKT-like",
]

# layer = "counts"

ad_tmp = adata[adata.obs[ct_key].isin(tcell_keys), av_genes + bv_genes].copy()

comp_results = []

for i in tqdm(range(n_perms)):
    ad_perm = ad_tmp.copy()

    ad_perm = permute_positions(ad_perm, rng=rng, sample_key="cc")

    # randomly scatter gene expression around t cells
    # ad_perm = permute_gene_expression(ad_perm, rng=rng, sample_key="cc")

    clone_df = identify_clonal_clusters(
        ad_perm,
        av_genes=av_genes,
        bv_genes=bv_genes,
        sample_key="cc",
        max_dist=100,
        min_cells=2,
        verbose=False,
        layer=layer,
    )
    avbv_cluster = merge_clonal_clusters(clone_df, verbose=False)
    ad_perm.obs["avbv_cluster"] = None
    ad_perm.obs.loc[avbv_cluster.index, "avbv_cluster"] = avbv_cluster
    filter_clonal_clusters_by_cell_type(
        ad_perm,
        ct_key="cell_type_l1.1",
        prohibited_combinations=[("CD4+", "CD8+")],
        in_key="avbv_cluster",
        out_key="avbv_cluster_filtered",
        prog_bar=False,
    )
    adata.obs["avbv_cluster_filtered_perm"] = None
    adata.obs.loc[ad_perm.obs_names, "avbv_cluster_filtered_perm"] = ad_perm.obs[
        "avbv_cluster_filtered"
    ].astype(str)
    mean_comp = compare_nhood_composition(
        adata, cc_key="avbv_cluster_filtered_perm"
    ).reset_index()
    mean_comp["iter"] = i
    comp_results.append(mean_comp)

save_path = (
    f"results/revision/nhood_comparison/nhood_comparison_random_permutation_{layer}.csv"
)
os.makedirs(os.path.dirname(save_path), exist_ok=True)
comp_results = pd.concat(comp_results, axis=0)
comp_results.to_csv(save_path)

  0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/13 [00:00<?, ?it/s]

nhoods with tcells not in cluster (unfiltered) (489493, 33)
nhoods with tcells in cluster (19567, 33)
nhoods with tcells not in cluster (0, 33)


IndexError: single positional indexer is out-of-bounds

In [28]:
mean_comp

,group,PapE,NEU,POD,PEC,MC,PC,DCT,pDC,glom. EC,...,gdT,unknown,MDC,MAIT,B,CD4+,CD8+,cycMNP,Tregs,iter
0,cluster\nnhood.,0.000000,0.000000,0.000682,0.000908,0.000289,0.002715,0.002452,0.000084,0.001737,...,0.003135,0.016791,0.131044,0.009515,0.075384,0.112351,0.075758,0.006774,0.018978,0
1,non-cluster\nnhood.,0.000084,0.000083,0.005599,0.005140,0.001345,0.010037,0.008292,0.000198,0.003555,...,0.002467,0.011756,0.085431,0.006093,0.047042,0.066610,0.042710,0.003452,0.008160,0


In [ ]:
save_path = (
    f"results/revision/nhood_comparison/nhood_comparison_random_permutation_{layer}.csv"
)
os.makedirs(os.path.dirname(save_path), exist_ok=True)
comp_results = pd.concat(comp_results, axis=0)
comp_results.to_csv(save_path)

## Plot results